In [1]:
from sentence_transformers import SentenceTransformer

d:\GitClones_new\AgenticAI_Study\codeBasics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
import logging
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
logging.getLogger("transformers").setLevel(logging.ERROR)

In [4]:
_model = SentenceTransformer("all-MiniLM-L6-v2")


def cosine_similarity(sentence1: str, sentence2: str) -> float:
    embeddings = _model.encode([sentence1, sentence2])
    a, b = embeddings[0], embeddings[1]
    similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.clip(similarity, 0.0, 1.0))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3055.84it/s]


In [13]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import HumanMessage

# llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)


@tool
def inventory_tool(product_name: str) -> str:
    """Check inventory availability for a given product name."""
    print(f"TOOL CALLED for: {product_name}")
    inventory = {
        "iPhone 15": "In Stock: Available Items = 2",
        "AirPods Pro": "Out of Stock: Available Items = 0",
        "MacBook Air M3": "In Stock: Available Items = 5"
    }
    return inventory.get(product_name, "Product not found in inventory.")

# Inventory Agent
agent = create_agent(
    model=llm,
    tools=[inventory_tool],
    system_prompt="""
    You are an inventory assistant.

    - If a question is out of scope which is not related to inventory then just say "Sorry, I can’t assist with that." No need to say anything else.
    
    When a user asks about a product, use the inventory_tool to fetch the inventory data.

    - Always call the inventory_tool with the full product name.
    - inventory_tool will return a dictionary which you need to parse to extract stock status, inventory items etc.
    - Respond with clear, concise information including:
        1. The stock status (e.g., "In Stock", "Out of Stock")
        2. The number of available items (if applicable)
    - If the product is not found, say: "The product is not available in our inventory."

    Never guess or hallucinate information. Do not respond unless the inventory_tool is called.
    Keep your response short and informative.
    """
)

def run(question: str) -> str:
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    return result["messages"][-1].content

In [14]:
from langsmith import Client, evaluate, traceable

In [15]:
@traceable
def target(inputs: dict) -> dict:
    question = inputs["question"]
    answer = run(question)
    return {"answer": answer}

client = Client()

dataset_name = "inventorydata"

In [16]:
if not client.has_dataset(dataset_name=dataset_name):
    client.create_dataset(dataset_name=dataset_name)

    client.create_examples(
        dataset_name=dataset_name,
        examples=[
            {
                "inputs": {"question": "What is the stock status of iPhone 15?"},
                "outputs": {"answer": "The iPhone 15 is currently in stock with 2 units available."},
            },
            {
                "inputs": {"question": "Is AirPods Pro available?"},
                "outputs": {"answer": "The AirPods Pro is currently out of stock. There are 0 available items."},
            },
            {
                "inputs": {"question": "How many iPhone 15 units are available?"},
                "outputs": {"answer": "The iPhone 15 is currently in stock with 2 units available."},
            },
            {
                "inputs": {"question": "Do you have Samsung Galaxy S23?"},
                "outputs": {"answer": "The product is not available in our inventory"},
            },
            {
                "inputs": {"question": "Can you tell me the recipe of Vada Pav?"},
                "outputs": {"answer": "Sorry, I can’t assist with that"},
            }
        ],
    )

def semantic_match(example, run):
    expected = example.outputs["answer"]
    actual = run.outputs["answer"]
    sim = cosine_similarity(expected, actual)
    return {
        "key": "semantic_match",
        "score": float(sim)
    }

In [17]:
evaluate(
    target,
    client=client,
    data=dataset_name,
    evaluators=[semantic_match],
    experiment_prefix="inventory_agent_evaluation_gpt-oss-20b"
)

View the evaluation results for experiment: 'inventory_agent_evaluation_gpt-oss-20b-83151204' at:
https://smith.langchain.com/o/eaa9c105-d6f3-4729-879b-f2d98f4fb232/datasets/de7faf1f-bd20-4d72-a557-7d49d9073c7e/compare?selectedSessions=3c92ed00-15fd-44f7-aea5-085b78459690




0it [00:00, ?it/s]

TOOL CALLED for: iPhone 15


1it [00:01,  1.10s/it]

TOOL CALLED for: Samsung Galaxy S23


...

TOOL CALLED for: iPhone 15


4it [00:03,  1.09it/s]

TOOL CALLED for: AirPods Pro


5it [00:05,  1.13s/it]


,inputs.question,outputs.answer,error,reference.answer,feedback.semantic_match,execution_time,example_id,id
0,How many iPhone 15 units are available?,In Stock: 2 units available.,None,The iPhone 15 is currently in stock with 2 uni...,0.684593,1.063640,0c3730dc-2507-47a6-83f5-91cb1db0bc18,01a12a8e-362f-74f3-9e56-13f9b01667e9
1,Do you have Samsung Galaxy S23?,The product is not available in our inventory.,None,The product is not available in our inventory,0.954644,0.986040,9c9b8394-64a3-4fcd-8f22-117d026cc1cf,01a12a8e-3a79-7c80-aa52-67f5ebbdf5c6
2,Can you tell me the recipe of Vada Pav?,"Sorry, I can’t assist with that.",None,"Sorry, I can’t assist with that",0.960514,0.497722,9cc589a7-792c-4eb7-b5e4-591c0c62d5e6,01a12a8e-3e6d-70b0-8af3-4b0748c5a51f
3,What is the stock status of iPhone 15?,In Stock – 2 items available.,None,The iPhone 15 is currently in stock with 2 uni...,0.580282,1.030457,9f32f242-db8b-47c5-b5d1-4d711d4a3b65,01a12a8e-407b-7172-9770-b00c8b8db3b8
4,Is AirPods Pro available?,Out of Stock. 0 items available.,None,The AirPods Pro is currently out of stock. The...,0.508419,1.106813,ef61ada2-9bf0-4a84-ab14-d22405c18523,01a12a8e-44bb-7173-8d14-92eb16a30a7e
